In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

In [ ]:
df = pd.read_csv("../data/processed/clean_data.csv")
df["ville"] = df["localisation"].str.split(",").str[0].str.strip()
print("Nombre d'annonces:", len(df))

In [ ]:
# Variables utilisees pour la prediction
features = ["surface", "chambres", "salles_de_bain", "etage"]

# Encoder la ville
top_villes = df["ville"].value_counts().head(20).index
df["ville_encode"] = df["ville"].apply(lambda x: x if x in top_villes else "Autre")
df = pd.get_dummies(df, columns=["ville_encode"])

# Ajouter les colonnes ville au features
ville_cols = [col for col in df.columns if col.startswith("ville_encode_")]
features = features + ville_cols

X = df[features]
y = df["prix"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, y_pred), 0), "DH")
print("R2 Score:", round(r2_score(y_test, y_pred), 3))

In [ ]:
# Exemple : appartement de 90m2, 2 chambres, 1 sdb, etage 2 a Casablanca
test = pd.DataFrame([{col: 0 for col in features}])
test["surface"] = 90
test["chambres"] = 2
test["salles_de_bain"] = 1
test["etage"] = 2
if "ville_encode_Casablanca" in features:
    test["ville_encode_Casablanca"] = 1

prix_predit = model.predict(test)[0]
print(f"Prix predit : {round(prix_predit, 0):,.0f} DH")

In [ ]:
import pickle

with open("../models/model_prix.pkl", "wb") as f:
    pickle.dump(model, f)

# Sauvegarder aussi les features pour Streamlit
with open("../models/features.pkl", "wb") as f:
    pickle.dump(features, f)

print("Modele sauvegarde.")

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder

# Ajouter plus de features
df["prix_par_m2"] = df["prix"] / df["surface"]

# Essayer un meilleur modele
model2 = RandomForestRegressor(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    random_state=42
)
model2.fit(X_train, y_train)
y_pred2 = model2.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, y_pred2), 0), "DH")
print("R2 Score:", round(r2_score(y_test, y_pred2), 3))